In [ ]:
import os
import cv2
import numpy as np
from deepface import DeepFace
from sklearn.svm import SVC
import joblib # Para guardar el modelo entrenado

# 1. Configuración
dataset_path = "dataset_emociones" # Cambia esto a tu ruta
model_name = "Facenet" # El modelo que usaremos para extraer características
target_size = (160, 160) # Tamaño que espera Facenet

X = [] # Aquí guardaremos los embeddings (datos)
y = [] # Aquí guardaremos las etiquetas (0, 1, 2...)
label_map = {} # Para recordar que 0 = feliz, 1 = enfadado...

# 2. Cargar imágenes y extraer embeddings
print("Cargando imágenes y extrayendo características...")
for i, class_name in enumerate(os.listdir(dataset_path)):
    class_path = os.path.join(dataset_path, class_name)
    if not os.path.isdir(class_path): continue
    
    label_map[i] = class_name
    print(f"Procesando clase: {class_name}")
    
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        
        try:
            # DeepFace.represent convierte la cara en una lista de números
            # enforce_detection=False es vital para que no falle si una foto es mala
            embedding = DeepFace.represent(img_path=img_path, model_name=model_name, enforce_detection=False)[0]["embedding"]
            X.append(embedding)
            y.append(i)
        except Exception as e:
            print(f"Error en {img_name}: {e}")

# 3. Entrenar el clasificador SVM
print("Entrenando el modelo SVM...")
clf = SVC(kernel='linear', probability=True) # linear suele ir bien para embeddings
clf.fit(X, y)

# 4. Guardar el modelo y el mapa de etiquetas
print("Guardando modelo...")
joblib.dump(clf, 'modelo_emociones.pkl')
joblib.dump(label_map, 'mapa_etiquetas.pkl')
print("¡Entrenamiento finalizado!")

In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time

# Cargamos solo las primeras 600 imagenes de cada emoción
MAX_IMAGES_PER_CLASS = 600 

print(f"Script 1: Extracción de Embeddings (MODO RÁPIDO: max {MAX_IMAGES_PER_CLASS} por clase)")

# --- Función LoadDataset (Adaptada de tu Deepface_kfold.ipynb) ---
def LoadDataset(folder, ext, max_per_class):
    nclasses = 0        # Contador de clases (emociones)
    nperclass = []      # Lista para guardar cuántas imágenes hay por clase
    classlabels = []    # Lista para los nombres de las clases
    X = []      # Lista para guardar los 'embeddings' (los datos)
    Y = []      # La lista de etiquetas (ej: 0, 1, 2) que van con X

    print(f"Cargando dataset desde: {folder}")
    
    # Obtener lista de clases (directorios)
    class_list = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
    
    # Recorre las carpetas de emociones (ej: 'happy', 'sad'...)
    for class_name in class_list:
        class_folder = os.path.join(folder, class_name)
            
        nclasses += 1   # Sumamos una clase al contador
        nsamples = 0    # Reseteamos el contador de imágenes para esta nueva clase
        print(f"\nCargando clase: {class_name} ({nclasses}/{len(class_list)})")

        # Recorre los archivos (imágenes) DENTRO de cada carpeta de emoción
        for file_name in os.listdir(class_folder):
            
            # Comprueba si ya hemos alcanzado el límite de imágenes
            if nsamples >= max_per_class:
                print(f"   ... Límite alcanzado ({max_per_class} imágenes)")
                break # Rompe el bucle de esta clase y pasa a la siguiente

            # Comprueba si el archivo es una imagen con la extensión correcta
            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name) # Ruta completa a la imagen
                try:
                    image = cv2.imread(image_path)  # Carga la imagen desde el disco
                    if image is None:
                        continue
                    
                    # Redimensiona la imagen al tamaño que espera 'Facenet' (160x160)
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    # Usa la red neuronal 'Facenet' para "mirar" la imagen (img1)
                    # y convertirla en un vector de 128 números (el 'embedding').
                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False # Asume que la imagen ya es una cara
                    )
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    X.append(img_embedding) # Añade el vector de 128 números a nuestra lista de datos 'X'
                    Y.append(nclasses - 1)  # Añade la etiqueta numérica (ej: 0 para 'angry') a nuestra lista 'Y'
                    nsamples += 1   # Suma 1 al contador de imágenes de esta clase
                    
                    # Imprime el progreso cada 100 imágenes
                    if nsamples % 100 == 0:
                        print(f"\r   ... procesadas {nsamples} imágenes", end="")
                
                except Exception as e:
                    # Ignora errores de 'represent' (ej. cara no encontrada)
                    pass

        print(f"\r   -> Clase '{class_name}' completada. Total: {nsamples} imágenes.")
        nperclass.append(nsamples)      # Guarda el total de esta clase
        classlabels.append(class_name)  # Guarda el nombre de esta clase

    # Convertimos las listas de Python en 'numpy arrays'
    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    if X.size == 0:
        return X, Y, 0, 0, 0, [], [], []

    n_samples, n_features = X.shape     # n_samples = ~4200 (600*7), n_features = 128
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]    # n_classes = 7

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- Configuración del Modelo DeepFace ---
model_name = "Facenet"
print(f"Construyendo modelo: {model_name}")
model = DeepFace.build_model(model_name)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- 2. Carga del Dataset ---
folder = "C:/Users/lucia/Downloads/train" 

# ¡Ahora pasamos el límite como argumento!
X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.png', MAX_IMAGES_PER_CLASS)

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")

# --- 3. GUARDAR LOS DATOS EXTRAÍDOS ---
if n_samples > 0:
    print("\nGuardando datos extraídos en archivos .pkl...")
    
    joblib.dump(X, 'embeddings_X.pkl')
    joblib.dump(Y, 'labels_Y.pkl')
    joblib.dump(class_names, 'emotion_class_names.pkl')
    
    print("¡Éxito! Archivos 'embeddings_X.pkl', 'labels_Y.pkl' y 'emotion_class_names.pkl' guardados.")
    print("Ahora puedes ejecutar '2_entrenar_svm.py'.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset ('folder') y la extensión ('.png').")

Script 1: Extracción de Embeddings (MODO RÁPIDO: max 600 por clase)
Construyendo modelo: Facenet
Dimensiones de entrada: (160, 160)
Cargando dataset desde: C:/Users/lucia/Downloads/train

Cargando clase: angry (1/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'angry' completada. Total: 600 imágenes.

Cargando clase: disgusted (2/7)
   -> Clase 'disgusted' completada. Total: 436 imágenes.

Cargando clase: fearful (3/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'fearful' completada. Total: 600 imágenes.

Cargando clase: happy (4/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'happy' completada. Total: 600 imágenes.

Cargando clase: neutral (5/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'neutral' completada. Total: 600 imágenes.

Cargando clase: sad (6/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'sa

In [7]:
import numpy as np
import joblib
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("Script 2: Entrenamiento del Modelo SVM (Rápido)")

# --- 1. Cargar datos pre-extraídos ---
try:
    print("Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...")
    X = joblib.load('embeddings_X.pkl')
    Y = joblib.load('labels_Y.pkl')
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características.")
except FileNotFoundError:
    print("Error: No se encontraron los archivos .pkl.")
    print("Por favor, ejecuta '1_extraer_embeddings.py' primero.")
    exit()

# --- 2. Entrenamiento del Modelo SVM ---
if X.shape[0] > 0:
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    print("Iniciando GridSearchCV para SVM...")
    t0 = time()
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,
        n_jobs=-1, # Usar todos los cores (ahora irá rápido)
        verbose=3  # Imprime el progreso
    )
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_)
    
    # 2.3 Guardar los 2 modelos finales para el prototipo
    final_model = clf.best_estimator_
    print("\nGuardando modelos finales...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')
    joblib.dump(scaler, 'emotion_scaler.pkl')
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.")
    print("¡Todo listo para ejecutar el prototipo!")
else:
    print("Error: Los datos cargados están vacíos.")

Script 2: Entrenamiento del Modelo SVM (Rápido)
Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...
Datos cargados: 4036 muestras, 128 características.

Entrenando Scaler (MinMaxScaler)...
Iniciando GridSearchCV para SVM...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
GridSearchCV terminado en 92.038s
Mejor estimador encontrado:
SVC(C=1000.0, class_weight='balanced', gamma=0.01, probability=True)

Guardando modelos finales...
¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.
¡Todo listo para ejecutar el prototipo!
